# DF-THC 10x: compact groups from the Hamiltonian

**Among the tables measured here, a good DF-THC table differs from a poor one less in its vectors than in how
its groups sit in orbital space: each group touches few orbitals and overlaps the others little. Building that
into the seed, from the Hamiltonian alone with no trained weights, reaches the scoring shape directly, with no
merge and no shrink.** On `dfthcsos` at (8, 143, 45) with 8-bit words that seed, rounded with a converged schedule,
gives **10.413×** the reference, checked here by the official verifier.

| Build | (R, B, C) | Qubits | Toffolis | `λ_eff` | `error_1` (mHa) | `error_2` (mHa) | Cost | Ratio |
|---|---|---:|---:|---:|---:|---:|---:|---:|
| Reference | | 730 | 93,237 | 49.300 | | | 3.356e9 | 1.000× |
| **Final, converged rounding** | **(8, 143, 45)** | **427** | **15,967** | **47.267** | **9.52** | **19.98** | **3.223e8** | **10.413×** |
| Open patterns | (9, 143, 40) | 428 | 16,237 | 47.106 | 11.82 | 19.98 | 3.274e8 | 10.251× |
| Coded bit 0 | (9, 143, 40) | 428 | 16,269 | 47.240 | 10.60 | 19.98 | 3.289e8 | 10.201× |
| Coded bit 0 | (9, 143, 41) | 428 | 16,356 | 47.158 | 12.90 | 19.98 | 3.301e8 | 10.165× |
| Coded bit 0 | (9, 143, 42) | 428 | 16,464 | 47.081 | 13.91 | 19.98 | 3.318e8 | 10.115× |
| Coded bit 0 | (9, 143, 43) | 428 | 16,532 | 47.118 | 13.29 | 19.98 | 3.334e8 | 10.065× |
| Coded bit 0 | (9, 143, 44) | 428 | 16,622 | 47.074 | 13.39 | 19.98 | 3.349e8 | 10.020× |
| Coded bit 0 | (9, 143, 46) | 428 | 16,801 | 47.163 | 13.18 | 19.98 | 3.391e8 | 9.894× |
| Plain 8 bits | (9, 143, 46) | 428 | 17,425 | 47.288 | 5.71 | 18.41 | 3.527e8 | 9.515× |

Cost is `λ_eff` × Toffolis × qubits, and the ratio is the reference's cost over the build's. `Λ` is the one-norm of
the factorization, `E_gap` how far its sum-of-squares energy sits below a fixed upper bound, and
`λ_eff = √(E_gap (2Λ − E_gap))`. The verifier rejects a build if `error_1` (the one-body residual) or `error_2` (the
two-body residual) exceeds 20 mHa, or if `Λ ≤ 49.0` or `E_gap ≤ 3.914898` Ha, so `E_gap` has a hard floor. All nine
builds use this notebook's own weights. §6 builds the final row live and records the other eight.

**Four new parts and two findings, in the order the pipeline uses them.**

1. **Compactness as the measure of a table.** Vector quality does not order tables: at the scoring shape,
   how well a single vector matches the target's top eigenspace is highest for the worst fit and lowest for
   the best. Compactness does order them there: a trained public group spreads over about 18 orbitals, a group
   trained from eigen-leaves over about 30, and the gap survives merging and shrinking (§1, `compact`). It is
   a guide, not a complete measure: a random table can be compact and still fit badly.
2. **A clustered Cholesky seed.** Pivoted Cholesky leaves are local by construction, and balanced k-means on
   their orbital profiles puts leaves that share orbitals in one group. The same L-BFGS recipe then reaches
   12.26 mHa after five rounds where the eigen seed reaches 19.09, and the same Adam schedule on 27 small
   groups reaches 11.23 mHa where eigen groups reach 17.08 (§2–3, `experiments/compact_seed.py`). Compactness
   can be measured before any fit, so the final tables start from the most compact of 500 k-means starts.
3. **Dealing rows before coded rounding.** Permuting rows inside a group leaves the Hamiltonian unchanged, so
   strong rows can be spread across the 16-address blocks that share a coded pattern. On the final table the
   heaviest block falls from 5.5× to 1.3× the mean block (§4, `experiments/deal_rows.py`).
4. **Open block patterns.** Coded bit 0 shares one affine pattern per block of 16 addresses, and this pipeline used
   to fix it at the block's first frozen angle, while nearly all of the block was still moving. Left open, each
   stage takes the cheapest pattern that agrees with what has frozen. The same dealt table then rounds to 19.46 mHa
   instead of 20.01, under the gate before any refit (§4, `experiments/quantize_rm.py`).
5. **Finding: the rounding was under-converged.** Progressive rounding, its L-BFGS stages and the fine start of its
   schedule are public (credited at the end), and michaelrustad already found that long L-BFGS early in the rounding
   recovers what short Adam refits do not. What is measured here is how far that goes. The schedule this pipeline ran froze the angles in 33 stages with 12k L-BFGS
   iterations each to 20% frozen and Adam after that. With 59 stages, twice as fine while the first 20% freeze, and
   60k L-BFGS iterations at every stage with no Adam, the same dealt (8, 143, 45) table rounds to 19.95 mHa instead of
   20.85, 16% less noise², and passes the gate with one group fewer than the 10.251× build. Which of those changes
   matters on its own is not tested (§4, §7).
6. **Finding: the budget-spending refit stopped at a kink.** Refitting the weights at fixed words to spend the rest
   of the error budget on `λ_eff` is phucthaiv02's idea, and manwithacat reached a constrained refit at fixed words
   independently the same week. This pipeline's penalty version stopped on the kink of its `E_gap` penalty; holding
   `error_2` and `E_gap` as constraints of an augmented Lagrangian moves past it, 47.31 to 47.16 on the (9, 143, 46)
   table, and §5 measures what each mHa of budget buys (`experiments/refit_al.py`).

**End to end, from the Hamiltonian to a verified circuit.**

| Step | What it does | Result at (8, 143, 45) | Origin | Section |
|---|---|---:|---|---|
| 1. Seed | Cholesky leaves clustered by orbital from the most compact of 500 starts, OMP vectors, least-squares weights | 814 mHa | **new** | §2 |
| 2. L-BFGS | eight rounds of 60k iterations at the shape | 14.53 mHa | public | §3 |
| 3. Adam | 1M steps, rate 1e-4 to 1e-6 | 14.16 mHa | public | §4 |
| 4. Deal | rows spread by strength across the 16-address blocks | same error | **new** | §4 |
| 5. Round | 8-bit progressive rounding with coded bit 0, block patterns left open, 59 stages of 60k L-BFGS iterations | 19.95 mHa | public idea, own code; open patterns **new**; schedule a finding | §4 |
| 6. Refine | exact gauge moves on the rounded table | `λ_eff` 48.14 | public, adapted | §5 |
| 7. Refit | one augmented-Lagrangian pass | `λ_eff` 47.27, `error_2` 19.98 mHa | public idea, own form | §5 |
| 8. Build | 8-bit circuits, then the official verifier | 15,967 Toffolis, 427 qubits, **10.413×** | public designs, own code | §6 |

This notebook builds the seeds and the final circuits live and runs the official verifier on them. Steps 2, 3
and 5 ran on an A100 (step 5 for 4.6 hours), and step 7 takes far longer on Kaggle's CPU than on a laptop, so their
results load from the dataset with their logs (`REFIT_LIVE = True` reruns the refit). The final build also passed
the official metric under seven random salts at full shots. The (9, 143, 40) table of the previous version reaches
10.251× with open patterns and 10.201× with locked ones, and tables at C = 41, 42, 43 and 44, from other k-means
starts, reach 10.165×, 10.115×, 10.065× and 10.020× (§6).

The weights are this notebook's own. The circuits and much of the method are not: the `D(w)` penalty (§1),
progressive rounding, coded bits, the lookup design and the loaders all come from public work, credited at
the end. §2, §4 and §5 print the source of the new parts as it runs.

What this does not show: that compactness is sufficient (at equal shape a random table is more compact
than the eigen seed and fits twice as badly, §1), which parts of the rounding schedule matter on their own (§4), or
anything beyond one molecule under one gate (§7).

In [ ]:
import functools, glob, hashlib, inspect, json, math, multiprocessing, os, re, subprocess, sys, time
from pathlib import Path

import numpy as np

try:
    import optax  # noqa: F401
except ImportError:                                  # the starter kit imports it, and not every image has it
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'optax'], check=True)

# The pipeline: the attached dataset on Kaggle, the sibling folder in the repository.
PIPELINE = next((Path(h).parent for h in [*glob.glob('/kaggle/input/**/dfthc_refit/fastfit.py', recursive=True),
                                           '../dfthc-refit/fastfit.py'] if Path(h).exists()), None)
if PIPELINE is None:
    raise FileNotFoundError('the pipeline (dfthc_refit/) was not found: attach this notebook\'s dataset')
sys.path.insert(0, str(PIPELINE))
sys.path.insert(0, str(PIPELINE / 'experiments'))
import paths                                         # finds the competition data; puts the verifier on sys.path
import build, fastfit, onebody
import deal_rows, df_init, refit_al, rmcode          # experiments/: the seed, the row deal, the constrained refit, open patterns
from dfthcsos import verify as V                     # the official chemistry verifier

import jax.numpy as jnp
from dfthc.train_df_thc_sga import Initialize

REFIT_LIVE = False                                   # True: rerun §5's solve (about 5 min on a fast laptop, far longer here)
BUILD_ALL = False                                    # True: build and verify all eight tables in §6, not only the final one
OUT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('build')
OUT.mkdir(exist_ok=True)
N = 76
R, B, C = 9, 143, 46                                 # the shape of the controls: 414 squares, 1,287 loaded vectors
C40, STARTS = 40, 500                                # the (9, 143, 40) table's copies per group (360 squares), and the k-means starts screened for it
R8, C45 = 8, 45                                      # the final table: eight groups of 45 copies, the same 360 squares
REFERENCE_COST = 3355599588.3155365                  # the sample submission's dfthcsos cost


def find(name: str) -> Path:
    """An input file, from the attached datasets on Kaggle or a local checkout."""
    for root in ['/kaggle/input', 'dataset']:   # Kaggle, then local
        hits = sorted(glob.glob(f'{root}/**/{name}', recursive=True))
        if hits:
            if len({hashlib.sha256(Path(h).read_bytes()).hexdigest() for h in hits}) > 1:
                raise ValueError(f'{name}: different files match: {hits}')
            return Path(hits[0])
    raise FileNotFoundError(f'{name}: attach the datasets listed in the last section')


ref = Initialize(str(paths.FCIDUMP), paths.UPPER_ENERGY)
ref = {k: (jnp.asarray(v, jnp.float64) if hasattr(v, 'dtype') and jnp.issubdtype(v.dtype, jnp.floating) else v)
       for k, v in ref.items()}
with np.load(paths.SHIFTS) as z:
    frozen = {k: jnp.asarray(z[k], jnp.float64) for k in ('Hsym_lower', 'H1_I')}
prob = fastfit.problem(ref, frozen, jnp.float64)     # the target in packed pair space
T, M, ii, jj = (np.asarray(x) for x in prob)
mult = np.where(ii == jj, 1.0, 2.0)                  # pair multiplicities
d = np.sqrt(mult)                                    # M = outer(mult, mult) weighs squared entries, so rows scale by √mult
Tw = d[:, None] * T * d[None, :]                     # error_2 = ‖AᵀA − Tw‖_F with the rows of A scaled by d
lam, Q = np.linalg.eigh(Tw)
Qtop = Q[:, np.argsort(-lam)[:R * C]]                # the top 414 eigenvectors


def table(name: str) -> tuple[np.ndarray, np.ndarray]:
    """Unit vectors U (R, B, N) and weights W (R, B, C) of a continuous table."""
    with np.load(find(name)) as z:
        U, W = z['U'].astype(float), z['W'].astype(float)
    return U / np.linalg.norm(U, axis=2, keepdims=True), W


def error2(U: np.ndarray, W: np.ndarray) -> float:
    """error_2 in mHa: the Frobenius residual of the squares against the target."""
    A = np.einsum('rbc,rbp->rcp', W, U[:, :, ii] * U[:, :, jj]).reshape(-1, len(ii)) * d
    return 1e3 * float(np.linalg.norm(A.T @ A - Tw))


def compact(U: np.ndarray, W: np.ndarray) -> tuple[float, float]:
    """Effective orbitals per group (exp of the entropy of its importance-weighted orbital profile) and
    the mean cosine between the groups' profiles."""
    importance = np.sum(W ** 2, axis=2)
    p = np.array([np.sum(i[:, None] * u ** 2, axis=0) for i, u in zip(importance, U)])
    p /= p.sum(axis=1, keepdims=True)
    orbitals = np.exp(-np.sum(p * np.log(p + 1e-300), axis=1))
    ov = p @ p.T
    n = np.sqrt(np.diag(ov))
    cos = ov / n[:, None] / n[None, :]
    k = len(p)
    return float(orbitals.mean()), float((cos.sum() - k) / (k * (k - 1)))


def vector_quality(U: np.ndarray) -> float:
    """Mean share of a vector's pair projector inside the target's top 414 eigenspace."""
    u = U.reshape(-1, N)
    return float(np.mean(np.sum((u[:, ii] * u[:, jj] * d @ Qtop) ** 2, axis=1)))


def dw(W: np.ndarray) -> float:
    return 0.5 * float(np.sum(np.abs(W).sum(axis=1) ** 2 - W.sum(axis=1) ** 2))


def fits(name: str) -> list[tuple[float, float]]:
    """(error_2, D) after each full-shape refit round of a long-run log."""
    pat = re.compile(r'^\s*\d+x\d+x\d+: error_2\s+[\d.]+ ->\s+([\d.]+) mHa\s+D\s+([\d.]+)')
    return [(float(a), float(b)) for a, b in (h.groups() for h in map(pat.match, find(name).read_text().splitlines()) if h)]


def curve(name: str) -> dict[int, float]:
    """iteration -> error_2 of a local CPU fit log."""
    pat = re.compile(r'^\s*(\d+): error_2\s+([\d.]+) mHa')
    out = {}
    for h in filter(None, map(pat.match, find(name).read_text().splitlines())):
        out.setdefault(int(h.group(1)), float(h.group(2)))    # a log may hold a second run after the first
    return out


def adam_run(name: str) -> tuple[float, float, float]:
    """(error_2 at the start, best error_2, final D) of an Adam log."""
    lines = find(name).read_text().splitlines()
    last = [h for h in map(re.compile(r'^\s*\d+: error_2\s+[\d.]+ mHa\s+D\s+([\d.]+).*best\s+([\d.]+)').match, lines) if h][-1]
    return float(re.search(r'error_2\s+([\d.]+)', lines[0]).group(1)), float(last.group(2)), float(last.group(1))


def rounded(name: str) -> float:
    """error_2 at the last stage of a progressive rounding log."""
    return [float(h.group(1)) for h in map(re.compile(r'^stage.*->\s+([\d.]+) mHa').match, find(name).read_text().splitlines()) if h][-1]


def show(obj, start: str | None = None, stop: str | None = None) -> None:
    """Print the source that ran, or the part of it between two markers."""
    src = inspect.getsource(obj)
    print(src[src.index(start) if start else 0:src.index(stop) if stop else len(src)])

## 1. What differs between tables

**The model.** A table (R, B, C) approximates the two-body part of the Hamiltonian by a sum of R·C squares. Group r
holds B shared unit vectors `u_rb` over the N = 76 orbitals, and copy c of the group weights them: its square is
`(Σ_b w_rbc n(u_rb))²`, with `n(u)` the occupation of orbital `u`. A row is one vector with its C weights. The
circuits load each vector as N − 1 = 75 Givens angles, the angle words, and the one-body part as N more rows, so
(8, 143, 45) loads 1,220 words. `D(w) = ½ Σ_rc [(Σ_b |w_rbc|)² − (Σ_b w_rbc)²]` is the part of `Λ` that grows when a
copy mixes signs; the fits here minimize `error_2 + κ D(w)`, and `κ` is that penalty's weight.

Rows of the table below are tables, in lineage order: small groups, merged in threes, shrunk or built at the scoring
shape, and a same-shape pair at the end. The public rows start from ahmedberatozer's (26, 60, 30) groups, merged and
shrunk by the pipeline of our earlier notebook,
[DF-THC: Merge Groups Before Dropping Them](https://www.kaggle.com/code/marculera/df-thc-merge-groups-before-dropping-them).

* **Vector quality:** the mean share of a loaded vector's pair projector, `u uᵀ` over orbital pairs, inside the
  target's top 414 eigenspace. The `R·C = 414` squares can only span that much, so this is how well a single vector
  is aimed.
* **Orbitals per group:** the exponential of the entropy of a group's orbital profile, each vector weighted by
  `Σ_c w²`: how many orbitals the group effectively lives on.
* **Overlap:** the mean cosine between two groups' profiles.

`error_2` of the public groups is the published 10-bit table, which is tuned to sit under the 20 mHa
gate, so its continuous fit is lower than the number shown. The four public rows are recorded, not computed here:
they were measured with the same functions as every other row (earlier versions computed them live), but
the tables derive from ahmedberatozer's factorization dataset, whose license is its own ("other": derived from the
competition data, its terms apply), so this notebook's Apache-2.0 dataset does not ship them.

In [ ]:
PUBLIC = {   # error_2, D(w), vector, orbitals, overlap: tables derived from ahmedberatozer's data, recorded, not shipped
    'public groups': ((26, 60, 30), 19.84, 17.42, 0.810, 17.7, 0.127), 'public, merged': ((9, 180, 90), 10.80, 10.84, 0.791, 32.3, 0.237),
    'public, merged, early shrink': ((9, 176, 84), 8.42, 4.84, 0.809, 31.5, 0.233), 'public, merged and shrunk': ((9, 143, 46), 13.51, 5.70, 0.943, 29.0, 0.224)}
rows = ['public groups', ('eigen groups, Adam', *table('eigen_27x60x30.npz')), ('clustered groups, Adam', *table('cluster_27x60x30.npz')),
        'public, merged', ('eigen, merged', *table('eigen_9x180x90.npz')),
        'public, merged, early shrink', ('eigen, merged, early shrink', *table('eigen_9x173x80.npz')),
        'public, merged and shrunk',
        ('eigen seed, direct', *table('eigen_9x143x46.npz')), ('clustered seed, direct', *table('cluster_9x143x46.npz')),
        ('eigen seed, same shape', *table('eigen_9x166x70.npz')), ('random start, same shape', *table('random_9x166x70.npz'))]
print(f"{'table':38s} {'shape':14s} {'error_2':>8s} {'D(w)':>7s} {'vector':>7s} {'orbitals':>9s} {'overlap':>8s}")
for row in rows:
    if isinstance(row, str):
        shape, e2, dw_, vec, orb, ov = PUBLIC[row]
        print(f"{row + ', recorded':38s} {str(shape):14s} {e2:8.2f} {dw_:7.2f} {vec:7.3f} {orb:9.1f} {ov:8.3f}")
        continue
    tag, U, W = row
    orb, ov = compact(U, W)
    print(f'{tag:38s} {str(W.shape):14s} {error2(U, W):8.2f} {dw(W):7.2f} {vector_quality(U):7.3f} {orb:9.1f} {ov:8.3f}')

Read down the last two columns. The public groups are compact from the start, and they stay well ahead of
the eigen lineage at every stage: merged (32 orbitals against 51) and early in the shrink (about 31 against
50, at nearby shapes). The eigen lineage was not shrunk further, so there is no later pair to compare. The
clustered rows, built for this compactness, are ahead of both: 15.6 orbitals at (27, 60, 30) and 20.4 at the
scoring shape.

`error_2` is not a comparison at those two stages: the public rows are the exact merge and a 12k refit, and a
120k refit takes (9, 176, 84) from 8.42 to 6.40 mHa.

Vector quality does not order the tables. At (9, 143, 46) the eigen seed has the highest and the worst fit,
and the clustered seed the lowest and the best. After the merge, the public lineage moves it far more than
compactness: from about 0.8 to 0.94, while the orbitals per group stay near 30.

The last two rows are the warning: at equal shape the random table is the more compact and fits more
than twice as badly. Compactness is not sufficient on its own.

Why would compactness matter? Two groups share a Gram block `G = (U_r U_qᵀ)^∘2`, so tables whose groups
overlap in orbital space are harder to fit independently. That is an explanation, not a result: what
is measured here is the correlation in this table and the control in §3.

## 2. The seed, from the Hamiltonian alone

Four steps, with no weights from any earlier table (the helpers are `experiments/df_init.py`'s):

1. Pivoted Cholesky of the weighted target gives 414 leaves. Each is a column of what the earlier leaves
   left unexplained, so it stays local: about 7 orbitals, against 44 for an eigen-leaf.
2. Balanced k-means on the leaves' orbital profiles gives 9 groups of 46 leaves.
3. Each group picks 143 shared vectors from its leaves' eigenvectors by simultaneous orthogonal
   matching pursuit.
4. Weights by weighted least squares, leaf by leaf.

Leaves are not a fit: 414 Cholesky leaves reproduce the target to about 7 mHa, against 3.5 for the top
414 eigen-leaves. The fit that follows is free to leave them. Round-robin assignment of the same
leaves is the control for the grouping.

K-means depends on its random start, and compactness can be measured before any fit, so the final tables add one
step. The cell below scores 500 starts by the orbitals per group of their leaves and builds the seed from the lowest,
which takes seconds: at (9, 143, 40), and at (8, 143, 45), whose 360 leaves are the same ones in eight groups. The
(9, 143, 46) seed and the controls use the first start.

In [ ]:
def pivoted_cholesky(K: int) -> np.ndarray:
    """K columns of the residual of Tw, each at its largest remaining diagonal."""
    diag, L = np.diag(Tw).copy(), np.zeros((len(Tw), K))
    for k in range(K):
        q = int(np.argmax(diag))
        L[:, k] = (Tw[:, q] - L[:, :k] @ L[q, :k]) / np.sqrt(diag[q])
        diag = np.maximum(diag - L[:, k] ** 2, 0)
    return L


def build_seed(leaves: list[np.ndarray], groups: list[list[int]]) -> tuple[np.ndarray, np.ndarray]:
    """Each group's shared vectors by matching pursuit over its leaves' eigenvectors, then each leaf's weights."""
    U, W = np.zeros((len(groups), B, N)), np.zeros((len(groups), B, len(groups[0])))
    for r, ks in enumerate(groups):
        cand = np.concatenate([np.linalg.eigh(leaves[k])[1].T for k in ks])
        U[r] = df_init.omp_bases(cand, np.stack([leaves[k][ii, jj] for k in ks]), ii, jj, mult, B, 0.0)
        pairs = (U[r][:, ii] * U[r][:, jj]) * mult[None, :]
        for c, k in enumerate(ks):
            W[r, :, c] = np.linalg.lstsq(pairs.T, leaves[k][ii, jj] * mult, rcond=None)[0]
    return U, W


K = R * C
L = pivoted_cholesky(K)
leaves = [df_init.unpack(L[:, k] / d, ii, jj) for k in range(K)]
effective = lambda ls: np.mean([np.exp(-np.sum((p := np.sum(l ** 2, 1) / np.sum(l ** 2)) * np.log(p + 1e-300))) for l in ls])   # noqa: E731
top = np.argsort(-lam)[:K]
print(f'{K} Cholesky leaves reproduce the target to {1e3 * np.linalg.norm(L @ L.T - Tw):.2f} mHa, the top {K} eigen-leaves to '
      f'{1e3 * np.sqrt(np.sum(np.delete(lam, top) ** 2)):.2f}; one leaf lives on {effective(leaves):.1f} orbitals, an eigen-leaf on '
      f'{effective([df_init.unpack(np.sqrt(lam[k]) * Q[:, k] / d, ii, jj) for k in top]):.1f}')
profile = np.array([np.sqrt(np.sum(l ** 2, axis=1)) for l in leaves])
profile /= np.linalg.norm(profile, axis=1, keepdims=True)
weight = np.array([np.sum(l ** 2) for l in leaves])
label = df_init.balanced_kmeans(profile, R, C, weight)
groups = [list(np.flatnonzero(label == r)) for r in range(R)]
U_c, W_c = build_seed(leaves, groups)
U_r, W_r = build_seed(leaves, [list(range(r, K, R)) for r in range(R)])
for tag, U, W in [('clustered', U_c, W_c), ('round-robin', U_r, W_r)]:
    orb, ov = compact(U, W)
    print(f'{tag:12s} seed error_2 {error2(U, W):8.2f} mHa, D(w) {dw(W):6.1f}, {orb:.1f} orbitals per group, overlap {ov:.3f}')
U_s, W_s = table('seed_9x143x46.npz')
print(f'the seed of the long run: same weights to {np.abs(W_c - W_s).max():.1e}, same vectors to {np.abs(np.abs(np.sum(U_c * U_s, axis=2)) - 1).max():.1e}')
K40 = R * C40                                        # pivoted Cholesky is greedy: the first 360 leaves are the leaves of a 360-leaf run
shares = np.array([np.sum(l ** 2, axis=1) for l in leaves[:K40]])


def seed_orbitals(lab: np.ndarray) -> float:
    """Orbitals per group of a leaf assignment, from the leaves alone: `compact` before any vectors exist."""
    g = np.array([shares[lab == r].sum(axis=0) for r in range(lab.max() + 1)])
    g /= g.sum(axis=1, keepdims=True)
    return float(np.exp(-np.sum(g * np.log(g + 1e-300), axis=1)).mean())


t0 = time.time()
screen = np.array([seed_orbitals(df_init.balanced_kmeans(profile[:K40], R, C40, weight[:K40], seed=s)) for s in range(STARTS)])
START40 = int(np.argmin(screen))
print(f'{STARTS} k-means starts at C = {C40} in {time.time() - t0:.0f} s: orbitals per group {screen.min():.2f} (start {START40}) to {screen.max():.2f}, '
      f'median {np.median(screen):.2f}, first start {screen[0]:.2f}')
label_40 = df_init.balanced_kmeans(profile[:K40], R, C40, weight[:K40], seed=START40)
U_c40, W_c40 = build_seed(leaves[:K40], [list(np.flatnonzero(label_40 == r)) for r in range(R)])
U_s40, W_s40 = table('seed_9x143x40.npz')
print(f'C = 40, seed error_2 {error2(U_c40, W_c40):.2f} mHa, the seed of the (9, 143, 40) table: same weights to {np.abs(W_c40 - W_s40).max():.1e}, '
      f'same vectors to {np.abs(np.abs(np.sum(U_c40 * U_s40, axis=2)) - 1).max():.1e}')
t0 = time.time()
screen8 = np.array([seed_orbitals(df_init.balanced_kmeans(profile[:K40], R8, C45, weight[:K40], seed=s)) for s in range(STARTS)])
START45 = int(np.argmin(screen8))
print(f'{STARTS} k-means starts at (8, 45) in {time.time() - t0:.0f} s: orbitals per group {screen8.min():.2f} (start {START45}) to {screen8.max():.2f}, '
      f'median {np.median(screen8):.2f}, first start {screen8[0]:.2f}')
label_45 = df_init.balanced_kmeans(profile[:K40], R8, C45, weight[:K40], seed=START45)
U_c45, W_c45 = build_seed(leaves[:K40], [list(np.flatnonzero(label_45 == r)) for r in range(R8)])
U_s45, W_s45 = table('seed_8x143x45.npz')
print(f'(8, 143, 45), seed error_2 {error2(U_c45, W_c45):.2f} mHa, the seed of the final table: same weights to {np.abs(W_c45 - W_s45).max():.1e}, '
      f'same vectors to {np.abs(np.abs(np.sum(U_c45 * U_s45, axis=2)) - 1).max():.1e}')
print()
show(df_init.balanced_kmeans)

## 3. The controls

**The short one isolates the grouping.** The same 414 leaves, the same basis selection and the same
5,000 L-BFGS iterations on a CPU, with only the assignment of leaves to groups changed. The eigen seed
is alongside as the baseline.

**The long one compares whole seeds.** 60k-iteration L-BFGS rounds at the shape, `κ` 5e-5, on an A100. The two
seeds differ in leaves, grouping, basis selection (an earlier scan of that alone moved the error by about 5%
at 500 iterations) and in the pair weighting of the truncation: the eigen seed scaled each pair by its
multiplicity where the `error_2` norm needs its square root, and §2's seed uses the square root. So this shows
that the seed is better, not which part is responsible. The eigen run stopped at five rounds when Colab
released the machine; the clustered run continued to eight.

**The third is the same idea at small groups.** Adam for 3M steps from 1e-3 to 1e-5 at (27, 60, 30), from a
clustered seed and from an eigen seed. The eigen groups then had a further warm restart. The clustered seed here
came from `df_init.py` and carries the same weighting as the eigen seed, so §2's correction is not in it;
it is also a different table from the direct one and has no verified build of its own.

In [ ]:
short = {'eigen, round-robin': curve('fit_eigen.log'), 'Cholesky, round-robin': curve('fit_chol_robin.log'),
         'Cholesky, clustered': curve('fit_chol_cluster.log')}
print(f"{'iteration':>9s}" + ''.join(f'{k:>24s}' for k in short))
for it in (1000, 2000, 3000, 4000, 5000):
    print(f'{it:9d}' + ''.join(f'{c[it]:24.2f}' for c in short.values()))
eigen, cluster = fits('t9.log')[1:], fits('t9c.log')[1:]       # [0] is the 2k warm-up
print(f"\n{'round':>5s} {'eigen':>9s} {'D(w)':>7s} {'clustered':>11s} {'D(w)':>7s}")
for k in range(len(cluster)):
    a = f'{eigen[k][0]:9.2f} {eigen[k][1]:7.2f}' if k < len(eigen) else f"{'':9s} {'':7s}"
    print(f'{k + 1:5d} {a} {cluster[k][0]:11.2f} {cluster[k][1]:7.2f}')
e0, e1, e1d = adam_run('eigen_27.log')
c0, c1, c1d = adam_run('cluster_27.log')
print(f'\n(27, 60, 30), Adam: eigen groups {e0:.2f} mHa after 3M steps, {e1:.2f} (D(w) {e1d:.2f}) after a warm restart; clustered groups {c1:.2f} (D(w) {c1d:.2f}) after 3M steps')
c43 = fits('tc43.log')[1:] + fits('tc43b.log')[1:]   # the run resumed after round two; each log opens with the 2k warm-up
print(f'\n(9, 143, 43), the same eight rounds: ' + ', '.join(f'{e:.2f}' for e, _ in c43) + f' mHa, D(w) {c43[-1][1]:.2f}')
print(f"(9, 143, 44), after the same eight rounds: the first k-means start {fits('tc44b.log')[-1][0]:.2f} mHa, the selected start {adam_run('adam_44.log')[0]:.2f}")
c42 = fits('t42.log')[1:]                            # [0] is the 2k warm-up
print(f'(9, 143, 42), its own screened start, the same eight rounds: ' + ', '.join(f'{e:.2f}' for e, _ in c42) + f' mHa, D(w) {c42[-1][1]:.2f}')
c41 = fits('t41.log')[1:]
print(f'(9, 143, 41), its own screened start, the same eight rounds: ' + ', '.join(f'{e:.2f}' for e, _ in c41) + f' mHa, D(w) {c41[-1][1]:.2f}')
c40 = fits('t40.log')[1:]
print(f'(9, 143, 40), the start screened in §2, the same eight rounds: ' + ', '.join(f'{e:.2f}' for e, _ in c40) + f' mHa, D(w) {c40[-1][1]:.2f}')
c45 = fits('t845.log')[1:]
print(f'(8, 143, 45), the start screened in §2, the same eight rounds: ' + ', '.join(f'{e:.2f}' for e, _ in c45) + f' mHa, D(w) {c45[-1][1]:.2f}')

The grouping alone is worth about 5 mHa at 5,000 iterations. Over the long run the gap between the two
seeds widens to 7 mHa, and `D(w)`, the sign-mixing term in `Λ`, halves. The clustered table at round
five is already below the 13.51 mHa of the (9, 143, 46) table reached by merging and shrinking public
groups, with no merge, no shrink and no public weights. It also stays compact: about 20 orbitals per
group at the end of the fit (§1), up from 12 in the seed.

The same recipe from a screened start stops at 12.70 mHa at C = 42, slightly below the 12.79 of the first start
at C = 43, which has one copy more, at 13.20 at C = 41 and at 13.75 at C = 40. With one group fewer, (8, 143, 45)
stops at 14.53. None of these runs has an eigen control of its own, so the controls above are all at (9, 143, 46).

## 4. Adam, rounding and coded bits

Adam and progressive rounding are public methods, and `quantize.py --coded` is this pipeline's implementation
of somejoint's coded bits. Dealing the rows and leaving block patterns open are new; the schedule is a finding. The
tables below run from (9, 143, 46), where the controls are, through (9, 143, 40) to the final (8, 143, 45).

* **Adam at the shape** (1M steps, `adam_train.py`, the optimizer of Low et al.) takes the table from
  12.08 to 11.69 mHa at C = 46 and from 13.75 to 13.39 at C = 40. The same pass takes the public (9, 143, 46)
  table from 13.51 to 13.04. Its learning-rate noise raises `E_gap` while the rate is high, so a table should be
  taken only once the rate has fallen; at the end of the C = 46 run it is back in place.
* **Rounding** (`quantize.py`) adds error roughly in quadrature, so the table reports
  `noise = √(final² − fit²)`.
* **Coded bit 0.** Over a block of 16 consecutive addresses, bit 0 of a coded word column is an affine function of
  the four low address bits: one of 32 patterns, a degree-1 Reed–Muller codeword, which the loader exploits to load
  fewer bits. The rounding must make every block's bits follow one pattern.
* **Dealing the rows.** Coded low bits share one affine pattern per 16 addresses, so a
  block of strong rows pays more. The fit leaves rows nearly sorted by strength, the worst layout.
  Permuting rows inside a group leaves the Hamiltonian unchanged, so `deal_rows.py` deals them
  across the blocks by strength first.
* **Open patterns.** `quantize.py --coded` fixes a block's affine pattern at the block's first frozen angle.
  `quantize_rm.py --codes 0:1:3-14` leaves it open: every stage takes, among the 32 affine patterns, the one of least
  curvature-weighted offset that agrees with the angles already frozen, until they pin it down. The loop printed
  below (`rmcode.residues`) does this one bit at a time, for any Reed-Muller code.
* **The schedule.** Each stage freezes a share of the angles at their rounded values and refits the rest. Up to
  version 9 every rounding here ran 33 stages, 12k L-BFGS iterations per stage to 20% frozen and 6k Adam steps per
  stage after that. The last rows vary that schedule on two dealt tables, with open patterns throughout.

In [ ]:
adam = {c: table(f'adam_9x143x{c}.npz') for c in (46, 44, 43, 42, 41, 40)} | {'8 x 45': table('adam_8x143x45.npz')}
fit = {c: error2(U, W) for c, (U, W) in adam.items()}
spread = lambda s: max(s[j * 16:(j + 1) * 16].sum() for j in range(-(-B // 16))) * -(-B // 16) / s.sum()     # noqa: E731
print(f"{'Adam table':>10s} {'error_2':>8s} {'D(w)':>6s} {'orbitals':>9s} {'overlap':>8s} {'heaviest block, as fit':>23s} {'dealt':>6s}")
for c, (U, W) in adam.items():
    orb, ov = compact(U, W)
    strength = np.sum(W ** 2, axis=2)
    print(f'{("C = " + str(c) if isinstance(c, int) else c):>10s} {fit[c]:8.2f} {dw(W):6.2f} {orb:9.1f} {ov:8.3f} {np.mean([spread(s) for s in strength]):23.2f} '
          f'{np.mean([spread(s[deal_rows.deal(s, 16)]) for s in strength]):6.2f}')
print(f"\n{'rounding':42s} {'fit':>7s} {'rounded':>8s} {'noise':>6s}")
for tag, f, log in [('public, κ 5e-5, plain 8 bits', 13.51, 'public_round_k5.log'), ('public, κ 2e-5, plain 8 bits', 12.81, 'public_round.log'),
                    ('clustered, 12.08, plain 8 bits', fits('t9c.log')[-1][0], 'cluster_round.log'),
                    ('clustered + Adam, plain 8 bits', fit[46], 'adam_round.log'), ('clustered + Adam, coded bit 0', fit[46], 'adam_coded_round.log'),
                    ('C = 44, selected start, coded bit 0', fit[44], 'coded_round_44.log'),
                    ('C = 43 + Adam, coded bit 0', fit[43], 'coded_round_43.log'),
                    ('C = 42, screened start, coded bit 0', fit[42], 'coded_round_42.log'),
                    ('C = 41, screened start, coded bit 0', fit[41], 'coded_round_41.log'),
                    ('C = 40, screened start, coded bit 0', fit[40], 'coded_round_40.log'),
                    ('C = 40, the same, open patterns', fit[40], 'open_round_40.log'),
                    ('C = 42, RM(2, 4) on bits 1 to 3 too', fit[42], 'plan_round_42.log')]:
    r = rounded(log)
    print(f'{tag:42s} {f:7.2f} {r:8.2f} {math.sqrt(r * r - f * f):6.2f}')


def stages(name: str) -> list[tuple[float, float, str]]:
    """(frozen share, error_2 after the stage, optimizer) for every stage of a rounding log."""
    pat = re.compile(r'^stage\s+\d+: frozen ([\d.]+)\s+error_2\s+[\d.]+ ->\s+([\d.]+) mHa\s+D\s+[\d.]+\s+(L-BFGS)?')
    return [(float(h.group(1)), float(h.group(2)), h.group(3) or 'Adam') for h in map(pat.match, find(name).read_text().splitlines()) if h]


SCHEDULES = [('C = 40, 33 stages, 12k', 40, 'open_round_40.log'), ('C = 40, 33 stages, 30k', 40, 'round_40_30k.log'),
             ('C = 40, 33 stages, 60k', 40, 'round_40_60k.log'), ('C = 40, 59 stages, 12k', 40, 'round_40_fine.log'),
             ('(8, 45), 33 stages, 12k', '8 x 45', 'round_845_12k.log'), ('(8, 45), 33 stages, 30k', '8 x 45', 'round_845_30k.log'),
             ('(8, 45), 59 stages, 60k, no Adam', '8 x 45', 'round_845_60k.log')]
print(f"\n{'schedule':32s} {'L-BFGS':>6s} {'Adam':>5s} {'noise² at 18%':>14s} {'final':>7s} {'noise²':>7s} {'in by 18%':>10s}")
for tag, c, log in SCHEDULES:
    st = stages(log)
    early = [e for share, e, _ in st if share <= 0.18][-1] ** 2 - fit[c] ** 2
    total = st[-1][1] ** 2 - fit[c] ** 2
    kinds = [k for *_, k in st[:-1]]                 # the last line is the closing check, not a stage
    print(f'{tag:32s} {kinds.count("L-BFGS"):6d} {kinds.count("Adam"):5d} {early:14.0f} {st[-1][1]:7.3f} {total:7.0f} {early / total:10.0%}')
print()
show(deal_rows.deal)
show(rmcode.residues, '    nblocks = ')

The compact table rounds no worse than the public one: its noise is about 14.3 mHa against 15.1–15.2 at both
of the public table's `κ`. Its `D(w)` is about the same as the public table's at equal `κ` (6.01 before
rounding and 6.17 after, against 5.70 and 6.08). That was not obvious in advance. A cheap proxy, weights-only
refitting after nearest rounding, predicted about 11% more noise on compact tables and was wrong in
direction: only a real progressive rounding answers this.

Coded bit 0 costs about 40 mHa² here and saves 624 Toffolis (10,154 → 9,530) in the angle circuits.

The noise is not a constant of the method. Across the coded tables it falls from 15.58 mHa at C = 46 to 14.56 at
C = 42, then rises to 14.77 at C = 41 and 14.86 at C = 40. So C = 42 rounds to 19.10 mHa from a fit level with
C = 43's, where C = 43's noise suggested about 19.54, and C = 40 rounds to 20.01, where C = 42's noise suggested
about 19.78. Quadrature is good for planning a shape, not for predicting its margin.

Why it moves is open, but C = 40 narrows it. Down to C = 42 the noise falls with C and with the overlap of the
finished table, and those tables cannot separate the two. C = 40 can: it has the fewest copies, its overlap rises to
0.072, and its noise rises with it. Ordered by overlap, the six tables run 14.77, 14.56, 14.86, 15.13, 15.24 and
15.58 mHa, rising at every step but the first, which points at overlap rather than C. Six tables is few and the
scatter between runs has not been measured, so this is a lead, not a result. The block spread does not explain it:
C = 43 down to C = 40 all deal to about 1.3. The three screened tables do finish as the most compact, at 17.2, 17.1
and 17.5 orbitals per group, though they also have the fewest copies.

All six of those tables lock their patterns. The open-pattern row starts from the same dealt table with the same
settings and schedule as the row above it, and differs only in that rule. Its noise is 14.12 mHa against 14.86, so
it rounds to 19.46 mHa, under the gate before any refit, which is about 22 mHa² of the 400 the gate allows. The gain
comes early, while blocks are still open: 0.36 mHa by 10% frozen and 0.52 by 40%, and it holds at 0.55 to the end.

The last row of that table is the same idea pushed further, and it failed; §7 gives the numbers.

**The schedule was the limit.** The second table rounds the same two dealt tables under longer schedules, all
with open patterns. On the C = 40 table, 30k L-BFGS iterations per stage instead of 12k take the noise² from 199 to
184 mHa², 60k to 176, and 59 stages at 12k, more than twice as fine to 20% frozen, to 182. So the rounding was not
converged, and the noise is not a property of the table alone.

Most of the noise enters early: 81–89% of the final noise² is in by 18% frozen, in every run. That is where longer
runs win. Past that point the lead shrinks: on C = 40 at 60k, 35 mHa² over the 12k run at 18% frozen and 23 at the
end; on (8, 143, 45) at 30k, 17 and 9. Those later stages use Adam. The final schedule uses L-BFGS at every stage,
59 stages of 60k iterations (4.6 hours on an A100), and keeps most of its lead: 42 mHa² at 18% frozen, 36 at the end.
It also has the finer stages, so this does not isolate Adam (§7).

On the (8, 143, 45) table the final schedule gives 19.95 mHa, where the 12k schedule gave 20.85 and 30k gave 20.63:
noise² 198 against 234, 16% less. That is the whole step from 10.251×. A lower rounded error at (9, 143, 40) buys
almost nothing: the 19.05 mHa rounding above, carried through the same refit and build, gives 10.250× against
10.251×, because the refit spends the budget on `λ_eff` and `E_gap` already sits on its floor. It pays as a smaller
shape. (8, 143, 45) fits worse than (9, 143, 40), 14.16 mHa against 13.39 after Adam, and rounds noisier, 234 mHa²
against 199 under the 12k schedule, so under that schedule it missed the gate by 0.85 mHa.

## 5. Spending the rest of the budget on `λ_eff`

`refit_weights.py`, our implementation of phucthaiv02's idea, minimizes `log λ_eff` with one-sided quadratic penalties on `E_gap` and `error_2`. On
these tables it stops within 2 iterations on the kink of the `E_gap` penalty: `λ_eff` falls to 47.31 but
`error_2` stays at 19.49 mHa, leaving the rest of the 20 mHa unspent. `refit_al.py` holds both
constraints with an augmented Lagrangian, so the solve can keep moving along the `E_gap` floor. The
two-body words stay the same; `w`, the offsets and the one-body rows change. The one-body rows are loaded
words too, so the circuits differ, but the refit does not change the Toffoli count.

The first table below is that trade on the (9, 143, 46) table, which rounded to 19.47 mHa. About a third of the
gain over the old refit's 47.31 needs no extra error: it was the kink. The budget above that buys about 0.19 of
`λ_eff` per mHa. The other direction is far dearer: asked for 19.00, the refit stalls at 19.30 with
`λ_eff` 48.44, because at fixed words the weights cannot buy back much rounding error.

At C = 40 with locked patterns the rounding misses the gate, at 20.01 mHa, and one pass still does the whole job.
`refine.py` leaves `E_gap` at 4.22, above its floor of about 3.92, and the solve's first step takes it to 3.93 and
`λ_eff` from 49.05 to 47.32 while `error_2` stays near 20.01. The rest of the pass has to pull `error_2` under the
target of 19.98 mHa, which it does, and ends at 47.24. That is the dear direction in a small dose: C = 41, with
0.43 mHa to spend, ends at 47.16.

With open patterns the same table reaches the solve with room to spend. `refine.py` again stops at `E_gap` 4.22 and
`λ_eff` 49.04, and the first two steps take them to 3.92 and 47.25 with `error_2` at 19.48. The rest of the pass
spends the remaining 0.50 mHa and ends at 47.11. So the 0.55 mHa from open patterns buys 0.13 of `λ_eff`, or 0.3% of
the score: `E_gap` already sits on its floor, and only `Λ` is left to move.

The final (8, 143, 45) table arrives with almost nothing to spend: it rounds to 19.95 mHa. `refine.py` stops at
`E_gap` 4.08 and `λ_eff` 48.14, and the pass takes `E_gap` to its floor, 3.92, and `λ_eff` to 47.27 while `error_2`
rises only to 19.98. So nearly all of its gain is the floor, not the budget. `REFIT_LIVE = True` reruns this pass,
which takes about 20 minutes on a laptop and far longer on Kaggle's CPU.

The one-body rows change here too. The verifier pairs rows with eigenvectors by a one-to-one assignment on |overlap|,
so their order is free, and `rmcode.refresh` deals them by |eigenvalue| across the blocks. On the C = 40 table `error_1`
ends at 11.82 mHa with open patterns against 10.60 with locked ones, well under its gate either way, so the C = 40
builds do not measure what dealing them is worth.

The solve is not always that clean. At C = 43 its line search failed after one step, with `error_2` at
19.56 mHa, `E_gap` still above the floor and both multipliers at zero, so neither constraint was holding it
there. A second pass from that result, after the solve's exact moves, ran normally and lowered
`λ_eff` from 47.27 to 47.12. The cause was not found. The cell below prints the passes from their logs.

In [ ]:
show(refit_al.solve, '    def lagrangian', '    z = np.concatenate')
print(f"{'error budget':>12s} {'error_2':>8s} {'λ_eff':>7s}")
for t in ('19.00', '19.49', '19.75'):
    m = json.loads(re.search(r'RESULT (\{.*\})', find(f'budget_{t}.log').read_text()).group(1))
    print(f"{t:>12s} {1e3 * m['error_2']:8.2f} {m['lambda_eff']:7.3f}")
with np.load(find('final_refit.npz')) as z:
    m = refit_al.rw.official({k: np.array(z[k]) for k in z.files}, 8)
print(f"{'19.98':>12s} {1e3 * m['error_2']:8.2f} {m['lambda_eff']:7.3f}")
with np.load(find('open_refit_40.npz')) as z:
    final = {k: np.array(z[k]) for k in z.files if k != 'codes'}
shipped = refit_al.rw.official(final, 8)
print(f"\nshipped (9, 143, 40) table: λ_eff {shipped['lambda_eff']:.3f}, error_2 {1e3 * shipped['error_2']:.2f} mHa, error_1 {1e3 * shipped['error_1']:.2f} mHa, D(w) {dw(final['W']):.2f}")
m = json.loads(re.findall(r'^refit (\{.*?\})', find('open_finish_40.log').read_text(), re.M)[-1])
print(f"C = 40, open patterns, its one pass, from its log: λ_eff {m['lambda_eff']:.3f}, error_2 {1e3 * m['error_2']:.2f} mHa, error_1 {1e3 * m['error_1']:.2f} mHa")
log845 = find('finish_845.log').read_text()
for step in ('refined', 'refit'):
    m = json.loads(re.findall(rf'^{step} (\{{.*?\}})', log845, re.M)[-1])
    print(f"(8, 143, 45), {step}, from its log: λ_eff {m['lambda_eff']:.3f}, E_gap {m['E_gap']:.3f}, error_2 {1e3 * m['error_2']:.2f} mHa, error_1 {1e3 * m['error_1']:.2f} mHa")
with np.load(find('refit_845.npz')) as z:
    final = {k: np.array(z[k]) for k in z.files if k != 'codes'}
shipped = refit_al.rw.official(final, 8)
print(f"shipped (8, 143, 45) table: λ_eff {shipped['lambda_eff']:.3f}, error_2 {1e3 * shipped['error_2']:.2f} mHa, error_1 {1e3 * shipped['error_1']:.2f} mHa, D(w) {dw(final['W']):.2f}")
for tag, name in [('C = 40, locked patterns, its one pass', 'refit_40.log'), ('C = 41, its one pass', 'refit_41.log'), ('C = 42, its one pass', 'refit_42.log'), ('C = 43, first pass', 'refit_43_pass1.log'), ('C = 43, second pass', 'refit_43_pass2.log')]:
    m = json.loads(re.findall(r'^RESULT (\{.*\})', find(name).read_text(), re.M)[-1])
    print(f"{tag}, from its log: λ_eff {m['lambda_eff']:.3f}, error_2 {1e3 * m['error_2']:.2f} mHa, error_1 {1e3 * m['error_1']:.2f} mHa")
if REFIT_LIVE:
    with np.load(find('refined_845.npz')) as z:
        deg = np.asarray(z['codes'], np.int64)
        data = {k: np.array(z[k]) for k in z.files if k != 'codes'}
    onebody.refresh = functools.partial(rmcode.refresh, deg=deg)   # the one-body rows keep their code and their dealt order
    t0 = time.time()
    cand, best = refit_al.solve(data, 19.98, 0.0005, 200, 3000, 3, log=lambda *x: None)
    print(f"the same solve here: λ_eff {best['lambda_eff']:.3f}, error_2 {1e3 * best['error_2']:.2f} mHa, error_1 {1e3 * best['error_1']:.2f} mHa ({time.time() - t0:.0f} s)")

## 6. Verified

The final (8, 143, 45) table is built here and goes through the official verifier. The other eight builds went
through the same verifier in the pipeline and are recorded below; `BUILD_ALL = True` rebuilds them. They are the
plain 8-bit (9, 143, 46) table, the same table with coded bit 0, dealt rows and the constrained refit, the
tables at C = 44, 43, 42, 41 and 40 carried through the same steps, and the C = 40 table again with open patterns.
The plain build used the old refit and leaves about 1.6 mHa of the budget unspent, so it is a lower bound for its
table. The coded builds also passed the official metric at full shots under random salts: seven each for the final
build, the open C = 40 build and at C = 40, 41 and 42, four each at C = 46 and C = 43, three at C = 44.

The two open-pattern builds go through `rmcode.build_family`, which differs from `tensorqrom.build_family` in one
way that matters here: circuits 0 to 3, which hold no coded bit, split their 35 output bits into five lookups of seven
bits, not six lookups of (7, 6, 6, 6, 5, 5), and circuits 4 to 14 come out byte-identical. On the C = 40 table that saves 34 Toffolis: built
the old way it has 16,271 Toffolis and scores 10.229×, so the step from 10.201× to 10.251× is about 0.028 from the
rounding and its refit and 0.021 from the layout. The layout owes nothing to open patterns.

From 10.251× to 10.413× the one change of shape is the dropped group. Eight groups of 45 copies hold the same 360
squares as nine of 40, but they need 143 fewer loaded vectors (1,220 angle words against 1,363), so the build loses
270 Toffolis and a qubit while `λ_eff` rises by 0.16.

In [ ]:
multiprocessing.set_start_method('spawn', force=True)   # the build's workers must not fork a process that has started JAX
os.environ['OPENBLAS_NUM_THREADS'] = '1'                # one BLAS thread per worker
RECORDED = {'plain, C = 46': (17425, 428, 47.288, 5.71, 18.41, 9.515), 'coded, C = 46': (16801, 428, 47.163, 13.18, 19.98, 9.894),
            'coded, C = 44': (16622, 428, 47.074, 13.39, 19.98, 10.020), 'coded, C = 43': (16532, 428, 47.118, 13.29, 19.98, 10.065),
            'coded, C = 42': (16464, 428, 47.081, 13.91, 19.98, 10.115), 'coded, C = 41': (16356, 428, 47.158, 12.90, 19.98, 10.165),
            'coded, C = 40': (16269, 428, 47.240, 10.60, 19.98, 10.201), 'open, C = 40': (16237, 428, 47.106, 11.82, 19.98, 10.251)}
tables = {'plain, C = 46': 'plain_refit.npz', 'coded, C = 46': 'final_refit.npz', 'coded, C = 44': 'final_refit_44.npz',
          'coded, C = 43': 'final_refit_43.npz', 'coded, C = 42': 'final_refit_42.npz', 'coded, C = 41': 'final_refit_41.npz',
          'coded, C = 40': 'final_refit_40.npz', 'open, C = 40': 'open_refit_40.npz', 'final, (8, 143, 45)': 'refit_845.npz'}
OPEN = {'open_refit_40.npz', 'refit_845.npz'}         # open-pattern tables: their codes go through rmcode.build_family


def build_open(name: str, out: Path) -> Path:
    """The final table's circuits: as build.build, with its words and codes through rmcode.build_family."""
    with np.load(find(name)) as z:
        arrays = {k: np.array(z[k]) for k in build.KEYS}
        words, edges, deg = (np.asarray(z[k], np.int64) for k in ('angle_words', 'tree_edges', 'codes'))
    out.mkdir(parents=True, exist_ok=True)
    np.savez(out / 'submission.npz', **arrays, tree_edges=edges, angle_words=words)
    build.write_kmx_family(rmcode.build_family(words, deg, 8, log=lambda *_: None), out, format='kmb')
    return out


results = {}
for k, (tag, name) in enumerate(tables.items()):
    if not BUILD_ALL and tag in RECORDED:
        continue
    t0 = time.time()
    sub = build_open(name, OUT / f'dfthcsos_{k}') if name in OPEN else build.build(find(name), OUT / f'dfthcsos_{k}', 8, 'log_depth', qrom='tensor')
    r = V.verify(sub)
    results[tag] = r
    print(f"{tag}: {r['status']} in {time.time() - t0:.0f} s: {r['toffoli_count']:,.0f} Toffolis ({r['resources']['angles_toffoli']:,.0f} in the angle circuits), "
          f"{r['num_qubits']} qubits, λ_eff {r['lambda_eff']:.3f}, error_1 {1e3 * r['metrics']['error_1']:.2f} mHa, error_2 {1e3 * r['metrics']['error_2']:.2f} mHa")
    print(f"       cost {r['score']:.4g} = {REFERENCE_COST / r['score']:.3f}x the reference")
print(f"\n{'recorded':16s} {'Toffolis':>9s} {'qubits':>7s} {'λ_eff':>7s} {'error_1':>8s} {'error_2':>8s} {'score':>8s}")
for tag, (t, q, l, e1, e2, x) in RECORDED.items():
    print(f'{tag:16s} {t:9,d} {q:7d} {l:7.3f} {e1:8.2f} {e2:8.2f} {x:7.3f}x')

## 7. What this shows, and what it does not

* **Shown:** on this Hamiltonian a seed built only from the target reaches a better table at the scoring
  shape than merging and shrinking trained public groups (12.08 against 13.51 mHa, and 11.69 against 13.04 after
  the same Adam pass), and it rounds no worse. Grouping leaves by orbital is responsible for part of that (§3,
  short control). Six fewer copies per group still pass the gate, and the verifier gives 10.251×. With the
  converged schedule one group fewer passes too, and the verifier gives 10.413×. The (9, 143, 40) table with
  locked patterns gives 10.201×, and tables at C = 41, 42, 43 and 44 reach 10.165×, 10.115×, 10.065× and 10.020×
  by the same steps.
* **Not shown:** which part of the schedule matters on its own. The final schedule changes three things at once:
  finer stages, five times the iterations, and no Adam stages. At (8, 143, 45) only it and the 12k and 30k schedules
  ran. The C = 40 rows separate finer stages from longer L-BFGS with the Adam stages kept, and show those stages giving
  back part of what longer L-BFGS gains early; no run drops the Adam stages alone.
* **Not shown:** how much screening the k-means start is worth. Each comparison here is one run per start. At
  C = 44 the first start ended L-BFGS at 13.27 mHa and a start selected for low overlap at 12.18; the first start
  was not carried further, so its final score is unknown. At C = 43 a second, lower-overlap start was no better
  (14.08 against 13.97 mHa at round two), so that table uses the first start. At C = 42 the most compact of 500
  starts, 11.88 orbitals per group against 13.41 for the first, ended at 12.70, below C = 43's 12.79 with one copy
  fewer. At C = 41 and C = 40 the most compact, 11.95 and 11.94 against 12.81 and 12.83, ended at 13.20 and 13.75.
  None of those three first starts was run, so the gain from screening is not measured.
* **Not shown:** what dealing the rows is worth. Every coded rounding here was dealt, so no run rounds the same
  table with coded bits as it was fit. The block spread in §4 is the evidence for it, not a final error.
* **Not shown:** what open patterns are worth in general. C = 40 is the one table with two finished runs that differ
  only in that rule, and the scatter between runs has not been measured.
* **Tried, and it did not pay, outside this notebook:** 9-bit rotation words, because the official cost formula picks
  the largest inner lookup whose temporary qubits fit under 9 × 75, and the qubits rise from 428 to 736 at every
  shape tried. A 46th copy per group at R = 8, which rounds to 19.59 mHa and builds to 10.4118×, a tie with 45 copies. A
  penalty on the rows' rounding sensitivity during the fit, which trades fit error for rounding noise nearly one for
  one: about 0.3 mHa net in prediction, not carried to a build.
* **Tried, and it did not pay:** codes above bit 0. RM(2, 4) on bits 1 to 3 of the later batches would take about
  450 more Toffolis off the loaders, with every codeword left open and the angles in the signed gauge from the first
  stage (`quantize_rm.py --open`). The relations hold on every block, but the C = 42 table rounds to 24.12 mHa, far
  over the gate, against 19.10 with locked coded bit 0 alone (§4).
* **Not shown:** why. Compactness correlates with fit quality and a controlled change improves it, but
  a random table shows it is not sufficient, and the long control does not separate the seed's parts. A long
  run of Cholesky leaves dealt round-robin would.
* **Not shown:** that merging is unnecessary in general. It remains exact and it gave the earlier
  tables; whether merging a compact seed improves on the direct build is not settled here.
* **One molecule, one gate:** FeMoCo76 at 20 mHa on `error_2` with the symmetry shift fixed. Nothing here
  tests another system, or the correlation-energy criterion of the DF-THC paper.

## Credits and references

This notebook and its pipeline build on public work. Each source keeps its own terms.

* **[ahmedberatozer, DF-THC Group Compression](https://www.kaggle.com/code/ahmedberatozer/quantum-ai-df-thc-group-compression):**
  the `D(w)` penalty, sensitivity-ordered progressive rounding, long refits, and the exact-gauge refinement of its
  `chemistry_refine.py`, which `refine.py` adapts (it in turn credits kozykappa and anazemcev, below). Its factorization
  dataset, [ahmedberatozer/df-thc-group-compression-factorization](https://www.kaggle.com/datasets/ahmedberatozer/df-thc-group-compression-factorization),
  holds the table recorded in §1: the compactness this notebook builds in was first seen there.
  [VQCC v8 components](https://www.kaggle.com/datasets/ahmedberatozer/vqcc-v8-components): the fine start of the freezing schedule.
* **[somejoint](https://www.kaggle.com/datasets/somejoint/qcc-our-chemistry-b143-c60-xor2):** coded low bits, an
  affine lowest bit per block of 16 addresses that the loader exploits; `quantize.py --coded` is our
  implementation.
* **[mrmeows, QCC open circuits and sources](https://www.kaggle.com/datasets/mrmeows/qcc-1375-open-circuits):** the exact
  lookup emitter's design that `tensorqrom.py` follows, and the one-body word polish that `onebody.py`
  extends. **heliosli:** J = 5 loaders. **[lightningv08](https://www.kaggle.com/code/lightningv08/bezout-schur-5-word-qrom):**
  five words per ANGLE circuit.
* **[kozykappa](https://www.kaggle.com/code/kozykappa/quantum-ai-complete-domain-affine-compilation), Kozykorpesh and Sumit Chongder:**
  the banked QROM and the signed Givens gauge. **kozykappa and Artem1410 (anazemcev):** the gauge
  rotation of the copy columns and the effective-norm refinement, which `refine.py` adapts. **haideptry**, via
  **[phucthaiv02](https://www.kaggle.com/code/phucthaiv02/qcc-verified-portfolio-df-thc-retraining-study):** the
  per-group orthogonal gauge; phucthaiv02 also refit the weights after rounding at fixed words, jointly (w, W), to
  spend the remaining error budget on `λ_eff`, the idea §5's refit puts in augmented-Lagrangian form.
* **[manwithacat, Hedge Wizardry](https://www.kaggle.com/code/manwithacat/vqcc-hedge-wizardry-notebook):** HW-03, a
  constrained `λ_eff` refit at fixed words, developed independently in the same week as §5's.
* **[michaelrustad](https://www.kaggle.com/code/michaelrustad/vqcc-r10-merged-group-chemistry-12-87-lb):** the `B ≥ 129` qubit
  cliff at 8 bits, L-BFGS refits of merged groups, and L-BFGS stages inside the progressive rounding (hybrid
  quantization), whose dose §4 raises. **k256net and Dmitrii Gluzdov:** the public factorization the
  measured groups descend from.
* **The hosts:** the verifier, kickmix and the starter kit.

The four new parts in the first cell were developed for this pipeline; the two findings rest on public methods,
credited above. A literature and notebook search found no earlier use of this seed, and a narrower search of public
notebooks found no earlier dealing of rows; neither is proof that none exists. For open patterns the search was narrower still: our saved public notebooks show nothing
like it, but somejoint's coded tables come without their rounding code, so how those patterns were chosen is
unknown. The closest idea we know is from steganography: wet paper codes write a message while some positions must
stay as they are, as the patterns here must agree with the frozen angles.

* Fridrich, Goljan, Lisoněk and Soukal, *Writing on wet paper*, IEEE Trans. Signal Process. 53 (2005) 3923–3935 (the closest idea to open patterns).
* Low et al., *Fast quantum simulation of electronic structure by spectrum amplification*, arXiv:2502.15882 (DF-THC-SOS, and the Adam recipe).
* Koch, Sánchez de Merás and Pedersen, *Reduced scaling in electronic structure calculations using Cholesky decompositions*, J. Chem. Phys. 118 (2003) 9481 (pivoted Cholesky of the two-electron integrals).
* Lee et al., *Even more efficient quantum computations of chemistry through tensor hypercontraction*, arXiv:2011.03494 (THC from Cholesky input).
* *Leveraging verifiable structure in quantum circuits to find more efficient compilations with agentic AI*, arXiv:2610.23832 (the competition's benchmark paper).

**Inputs:** the competition data, the hosts' starter kit
([sohier/verifiable-circuit-compilation-starter-kit](https://www.kaggle.com/datasets/sohier/verifiable-circuit-compilation-starter-kit)),
and this notebook's dataset ([marculera/dfthc-compact-pipeline](https://www.kaggle.com/datasets/marculera/dfthc-compact-pipeline)):
the pipeline (`dfthc_refit/`), the tables and the run logs.